# Module 3: Build your own training loop

**Project:** write a reusable training and validation loop, make a model overfit on purpose, fix it, and track experiments with a 10-run sweep.

**Done when:**
- your `fit()` function trains, validates, saves the best checkpoint, and can stop early,
- your regularized model beats the overfit baseline on validation data,
- you ran a 10-run sweep and can explain **three changes that improved the validation score**.

**How to use this notebook**
1. Turn on a GPU: *Runtime > Change runtime type > T4 GPU*, then save your own copy (*File > Save a copy in Drive*).
2. Run cells from top to bottom. Fill in every `YOUR CODE HERE`. The **Self-check** cells tell you if you got it right.
3. Weights & Biases (wandb) is optional. Everything works without it. Turn it on in Step 7 if you want the online dashboard.
4. Stuck? Use the hint ladder: docs, then the hints below each step, then the course forum. Only after your own attempt, open the solution page.

## Step 1: Setup and data (given)

We use only **2,000 training images** on purpose. A small training set is how you make overfitting easy to see. A separate 2,000 images form the **validation set**, and the original 10,000 test images stay untouched until the very end.

In [ ]:
import copy, random, time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

def set_seed(seed=0):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

tfm = transforms.ToTensor()
full_train = datasets.FashionMNIST("data", train=True, download=True, transform=tfm)
test_ds = datasets.FashionMNIST("data", train=False, download=True, transform=tfm)

perm = torch.randperm(len(full_train), generator=torch.Generator().manual_seed(0)).tolist()
train_ds = Subset(full_train, perm[:2000])
val_ds = Subset(full_train, perm[2000:4000])
print("train:", len(train_ds), " validation:", len(val_ds), " test:", len(test_ds))

def make_loaders(batch_size=64):
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            DataLoader(val_ds, batch_size=512),
            DataLoader(test_ds, batch_size=512))

def make_model(hidden=512, dropout=0.0):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, hidden), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(hidden, 10),
    ).to(device)

loss_fn = nn.CrossEntropyLoss()

## Step 2: Evaluate

Write `evaluate(model, loader, loss_fn, device)`. It returns `(average loss, accuracy)` over **every example** in the loader.

- Put the model in evaluation mode and turn off gradient tracking.
- `loss_fn` gives the *mean* loss of one batch. To get the mean over all examples, multiply each batch's loss by the batch size, add them up, and divide by the total number of examples at the end. (The last batch can be smaller.)

*Hint:* `loss.item() * y.size(0)`, and `(logits.argmax(1) == y).sum().item()` counts correct predictions.

In [ ]:
def evaluate(model, loader, loss_fn, device):
    """Return (mean loss per example, accuracy) over the whole loader."""
    model.eval()
    # YOUR CODE HERE
    return 0.0, 0.0

### Self-check: evaluate

In [ ]:
from torch.utils.data import TensorDataset

torch.manual_seed(0)
_X = torch.randn(7, 4); _y = torch.tensor([0, 1, 2, 0, 1, 2, 0])
_toy = nn.Linear(4, 3)
_loader = DataLoader(TensorDataset(_X, _y), batch_size=3)     # batches of 3, 3 and 1
_expected_loss = loss_fn(_toy(_X), _y).item()
_expected_acc = (_toy(_X).argmax(1) == _y).float().mean().item()
_loss, _acc = evaluate(_toy, _loader, loss_fn, "cpu")
assert abs(_loss - _expected_loss) < 1e-5, f"Loss is {_loss:.5f} but should be {_expected_loss:.5f}. Average over examples (weight each batch by its size), not over batches."
assert abs(_acc - _expected_acc) < 1e-6, f"Accuracy is {_acc:.4f} but should be {_expected_acc:.4f}."
print("evaluate() check passed.")

## Step 3: Train for one epoch

Write `train_one_epoch(model, loader, loss_fn, optimizer, device)`. It returns `(average loss, accuracy)` over the epoch, computed the same way as in `evaluate`.

- Start with `model.train()`. This matters later, because dropout is only active in training mode.
- Per batch: forward pass, loss, `zero_grad`, `backward`, `step`. You know these from Module 1.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    """Train for one pass over the loader. Return (mean loss per example, accuracy)."""
    model.train()
    # YOUR CODE HERE
    return 0.0, 0.0

### Self-check: train_one_epoch

In [ ]:
torch.manual_seed(0)
_X = torch.randn(200, 2); _y = (_X[:, 0] + _X[:, 1] > 0).long()     # an easy, separable problem
_loader = DataLoader(TensorDataset(_X, _y), batch_size=20, shuffle=True)
_toy = nn.Linear(2, 2)
_before = _toy.weight.detach().clone()
_opt = torch.optim.SGD(_toy.parameters(), lr=0.5)
_first = train_one_epoch(_toy, _loader, loss_fn, _opt, "cpu")
for _ in range(14):
    _last = train_one_epoch(_toy, _loader, loss_fn, _opt, "cpu")
assert not torch.equal(_before, _toy.weight.detach()), "The weights never changed. Did you call optimizer.step()?"
assert _last[0] < _first[0], f"The loss did not go down ({_first[0]:.3f} to {_last[0]:.3f}). Check the order: forward, loss, zero_grad, backward, step."
assert _last[1] > 0.9, f"Accuracy is only {_last[1]:.2f} after 15 epochs on an easy problem. Something is wrong in the loop."
assert _toy.training, "train_one_epoch should leave the model in training mode (call model.train())."
print("train_one_epoch() check passed.")

## Step 4: The full `fit()` loop

Write `fit(...)`. This is the heart of the module. For every epoch it must:

1. train for one epoch, then evaluate on the validation set,
2. record `train_loss, train_acc, val_loss, val_acc, lr` in the `history` dictionary (lists, one value per epoch),
3. call `log_fn(epoch, metrics)` if one was given (this is how experiment trackers plug in),
4. **if the validation loss improved**, save the model weights to `ckpt_path`, remember `best_epoch` and `best_val_loss`, and reset the counter,
5. **otherwise**, add 1 to a counter, and stop if `patience` is set and the counter has reached it,
6. call `scheduler.step()` once per epoch, if a scheduler was given. Record the learning rate **before** you step the scheduler.

After the loop, **load the best checkpoint back into the model**, add `best_epoch` and `best_val_loss` to `history`, and return it.

*Hints:* "improved" means strictly lower than the best so far (start from `float("inf")`). Save with `torch.save(model.state_dict(), ckpt_path)` and restore with `model.load_state_dict(torch.load(ckpt_path, map_location=device))`. The current learning rate is `optimizer.param_groups[0]["lr"]`.

In [ ]:
def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs, device,
        patience=None, ckpt_path="best.pt", scheduler=None, log_fn=None):
    """Train and validate. Return history with the best epoch, restoring the best weights."""
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    best_val_loss, best_epoch, bad_epochs = float("inf"), -1, 0
    # YOUR CODE HERE
    history["best_epoch"] = best_epoch
    history["best_val_loss"] = best_val_loss
    return history

### Self-check: fit

In [ ]:
import os
torch.manual_seed(0)
_Xtr = torch.randn(200, 2); _ytr = (_Xtr[:, 0] + _Xtr[:, 1] > 0).long()
_Xva = torch.randn(100, 2); _yva = (_Xva[:, 0] + _Xva[:, 1] > 0).long()
_tr = DataLoader(TensorDataset(_Xtr, _ytr), batch_size=20, shuffle=True)
_va = DataLoader(TensorDataset(_Xva, _yva), batch_size=50)

# 1) a normal run: 10 epochs, no early stopping
_m = nn.Linear(2, 2); _calls = []
_h = fit(_m, _tr, _va, loss_fn, torch.optim.SGD(_m.parameters(), lr=0.5), epochs=10, device="cpu",
         ckpt_path="_t1.pt", log_fn=lambda e, m: _calls.append(e))
assert all(len(_h[k]) == 10 for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr"]), "Each history list needs one value per epoch (10)."
assert _calls == list(range(10)), "log_fn must be called once per epoch with the epoch number 0, 1, 2, ..."
assert os.path.exists("_t1.pt"), "No checkpoint was saved to ckpt_path."
assert _h["best_epoch"] == int(np.argmin(_h["val_loss"])), "best_epoch should be the epoch with the lowest validation loss."
assert abs(_h["best_val_loss"] - min(_h["val_loss"])) < 1e-9, "best_val_loss should equal the lowest validation loss."
_check_loss, _ = evaluate(_m, _va, loss_fn, "cpu")
assert abs(_check_loss - _h["best_val_loss"]) < 1e-5, "After fit(), the model must hold the BEST weights. Load the checkpoint at the end."

# 2) early stopping: with lr = 0 nothing improves, so patience=2 must stop after 3 epochs
_m = nn.Linear(2, 2)
_h = fit(_m, _tr, _va, loss_fn, torch.optim.SGD(_m.parameters(), lr=0.0), epochs=20, device="cpu",
         patience=2, ckpt_path="_t2.pt")
assert len(_h["val_loss"]) == 3, f"With patience=2 and no improvement, training should stop after 3 epochs, but ran {len(_h['val_loss'])}."

# 3) a scheduler must be stepped once per epoch
_m = nn.Linear(2, 2); _o = torch.optim.SGD(_m.parameters(), lr=1.0)
_s = torch.optim.lr_scheduler.StepLR(_o, step_size=1, gamma=0.5)
_h = fit(_m, _tr, _va, loss_fn, _o, epochs=3, device="cpu", scheduler=_s, ckpt_path="_t3.pt")
assert _h["lr"] == [1.0, 0.5, 0.25], f"Recorded learning rates should be [1.0, 0.5, 0.25] but are {_h['lr']}. Record lr before calling scheduler.step()."
for _f in ["_t1.pt", "_t2.pt", "_t3.pt"]: os.remove(_f)
print("fit() check passed. You now own a real training loop!")

## Step 5: Overfit on purpose

Now use your loop. This cell is given: a big model (two layers of 512 units), no regularization, trained for 30 epochs on only 2,000 images. Watch the validation loss: it should fall and then rise.

In [ ]:
set_seed(0)
train_dl, val_dl, test_dl = make_loaders(64)
model_over = make_model(hidden=512, dropout=0.0)
opt = torch.optim.Adam(model_over.parameters(), lr=1e-3)
hist_over = fit(model_over, train_dl, val_dl, loss_fn, opt, epochs=30, device=device, ckpt_path="over.pt")

def plot_history(hist, title):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(hist["train_loss"], label="train"); ax[0].plot(hist["val_loss"], label="validation")
    ax[0].axvline(hist["best_epoch"], ls="--", c="gray"); ax[0].set_title(title + ": loss"); ax[0].legend()
    ax[1].plot(hist["train_acc"], label="train"); ax[1].plot(hist["val_acc"], label="validation")
    ax[1].set_title(title + ": accuracy"); ax[1].legend()
    plt.show()

plot_history(hist_over, "No regularization")
gap_over = hist_over["train_acc"][-1] - hist_over["val_acc"][-1]
print(f"best epoch {hist_over['best_epoch']}, train acc {hist_over['train_acc'][-1]:.3f}, "
      f"val acc {hist_over['val_acc'][-1]:.3f}, gap {gap_over:.3f}")

### Self-check: overfitting

In [ ]:
assert gap_over > 0.08, f"Expected a clear gap between train and validation accuracy, but the gap is {gap_over:.3f}. Is fit() correct?"
assert hist_over["val_loss"][-1] > hist_over["best_val_loss"], "Validation loss should have risen after its best epoch (overfitting)."
print("You can now see overfitting in the curves.")

**Question:** at which epoch did the validation loss stop improving? What happens to the training accuracy after that? Write your answer here (double-click to edit):

*Your answer:*

## Step 6: Fix it with two regularization techniques

Pick **at least two** of these by editing the three settings below, then run the cell:

- **Dropout** (`dropout`): randomly switches off units during training. Try 0.3 to 0.5.
- **Weight decay** (`weight_decay`): gently pulls weights toward zero. Try 0.01 to 0.1.
- **Early stopping** (`patience`): stop when validation loss has not improved for this many epochs. Try 5.

Change them from their defaults (0.0, 0.0, None), and compare with the baseline.

In [ ]:
dropout = 0.0        # YOUR CODE HERE: try 0.3 to 0.5
weight_decay = 0.0   # YOUR CODE HERE: try 0.01 to 0.1
patience = None      # YOUR CODE HERE: try 5

set_seed(0)
train_dl, val_dl, test_dl = make_loaders(64)
model_reg = make_model(hidden=512, dropout=dropout)
opt = torch.optim.AdamW(model_reg.parameters(), lr=1e-3, weight_decay=weight_decay)
hist_reg = fit(model_reg, train_dl, val_dl, loss_fn, opt, epochs=30, device=device,
               patience=patience, ckpt_path="reg.pt")
plot_history(hist_reg, "With regularization")
gap_reg = hist_reg["train_acc"][-1] - hist_reg["val_acc"][-1]
print(f"stopped after {len(hist_reg['val_loss'])} epochs, best val loss {hist_reg['best_val_loss']:.3f} "
      f"(baseline {hist_over['best_val_loss']:.3f}), gap {gap_reg:.3f} (baseline {gap_over:.3f})")

### Self-check: regularization

In [ ]:
_used = (dropout > 0) + (weight_decay > 0) + (patience is not None)
assert _used >= 2, "Use at least two regularization techniques: change two or more of dropout, weight_decay, patience."
assert gap_reg < gap_over or hist_reg["best_val_loss"] < hist_over["best_val_loss"], \
    "The regularized model should have a smaller train/validation gap or a lower best validation loss than the baseline. Try stronger settings."
print("Regularization check passed.")

**Question:** which two techniques did you use, and what changed in the curves? Write your answer here:

*Your answer:*

## Step 7: Experiment tracking

An experiment tracker records the settings and metrics of every run, so you can compare runs and reproduce the best one. **Weights & Biases** (wandb) is a popular free option.

- To use it, create a free account at wandb.ai, set `USE_WANDB = True`, and paste your API key when asked.
- To skip it, leave `USE_WANDB = False`. Your results are still kept in memory and shown as a table in Step 8.

In [ ]:
USE_WANDB = False   # set to True after creating a free account at wandb.ai

if USE_WANDB:
    try:
        import wandb
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
        import wandb
    wandb.login()
else:
    print("wandb is OFF. Results are kept in memory and shown as tables and plots.")

Now write `run_experiment(config)`. It trains **one** model described by a settings dictionary and returns a result.

Fill in the lines marked `YOUR CODE HERE`:
1. Build the model with `make_model(config["hidden"], config["dropout"])`.
2. Build an `AdamW` optimizer with `lr=config["lr"]` and `weight_decay=config["weight_decay"]`.
3. Define `log_fn`: if `use_wandb` is True it should call `wandb.log(metrics, step=epoch)`, otherwise leave it as `None`.
4. Call `fit(...)` with `epochs=config["epochs"]` and `patience=config["patience"]`, and keep the result in `hist`.

In [ ]:
def run_experiment(config, use_wandb=False):
    """Train ONE model described by `config`. Return a result dictionary."""
    set_seed(config.get("seed", 0))
    train_dl, val_dl, _ = make_loaders(config["batch_size"])

    # YOUR CODE HERE: model, optimizer and log_fn
    model = None
    optimizer = None
    log_fn = None

    if use_wandb:
        wandb.init(project="dl-startups-module3", config=config)

    # YOUR CODE HERE: call fit(...) and store its result in hist
    hist = None

    if use_wandb:
        wandb.finish()
    if model is None or optimizer is None or hist is None:
        raise NotImplementedError("Fill in the YOUR CODE HERE lines in run_experiment().")

    best = hist["best_epoch"]
    return {"config": config, "val_acc": hist["val_acc"][best], "val_loss": hist["best_val_loss"],
            "best_epoch": best, "history": hist, "model": model}

### Self-check: run_experiment

In [ ]:
_cfg = {"lr": 1e-3, "batch_size": 64, "hidden": 128, "dropout": 0.2, "weight_decay": 0.01, "epochs": 2, "patience": None}
_r = run_experiment(_cfg, use_wandb=USE_WANDB)
assert 0.0 < _r["val_acc"] <= 1.0, f"val_acc should be between 0 and 1, but is {_r['val_acc']}."
assert len(_r["history"]["val_loss"]) == 2, "fit() should run for config['epochs'] epochs."
assert "model" in _r and "best_epoch" in _r, "The result needs model and best_epoch."
print(f"run_experiment() check passed. Quick 2-epoch run: val accuracy {_r['val_acc']:.3f}")

## Step 8: A 10-run sweep

A **sweep** runs many settings and compares them. Try 5 learning rates and 2 batch sizes: 10 runs. Change nothing else, so you know what caused any difference.

Write the loop: for every learning rate and every batch size, build `config = {**base, "lr": lr, "batch_size": bs}`, call `run_experiment(config, use_wandb=USE_WANDB)`, and append the result to `results`. Runs take only a few seconds each.

In [ ]:
base = {"hidden": 512, "dropout": 0.3, "weight_decay": 0.01, "epochs": 15, "patience": 5}
lrs = [3e-2, 1e-2, 3e-3, 1e-3, 3e-4]
batch_sizes = [32, 128]
results = []

# YOUR CODE HERE: loop over lrs and batch_sizes, run each experiment, and append the result to `results`


### Self-check: sweep

In [ ]:
assert len(results) == 10, f"Expected 10 runs (5 learning rates x 2 batch sizes) but found {len(results)}."
assert len({(r["config"]["lr"], r["config"]["batch_size"]) for r in results}) == 10, "Every run should use a different (lr, batch_size) pair."
print("Sweep check passed.")

Now look at the results. This cell is given.

In [ ]:
import pandas as pd

table = pd.DataFrame([{"lr": r["config"]["lr"], "batch_size": r["config"]["batch_size"],
                       "val_acc": round(r["val_acc"], 4), "val_loss": round(r["val_loss"], 4),
                       "best_epoch": r["best_epoch"]} for r in results]).sort_values("val_acc", ascending=False)
display(table.reset_index(drop=True))

for bs, group in table.groupby("batch_size"):
    group = group.sort_values("lr")
    plt.plot(group["lr"], group["val_acc"], marker="o", label=f"batch size {bs}")
plt.xscale("log"); plt.xlabel("learning rate"); plt.ylabel("validation accuracy"); plt.legend()
plt.title("Learning rate sweep"); plt.show()

**Questions** (write your answers below):
1. Which setting won, and why do you think it did?
2. Look at the differences between your top few runs. With only 2,000 validation images, accuracy is only known to about plus or minus 0.8 percentage points. Are the top runs really different?
3. Name **three changes** from this module that improved your validation score.

*Your answers:*

## Step 9: The final exam (use the test set once)

Take the best run from the sweep and score it on the **test set**, once. This is your honest final number. This cell is given.

In [ ]:
best = max(results, key=lambda r: r["val_acc"])
print("Best config:", {k: best["config"][k] for k in ("lr", "batch_size")})
test_loss, test_acc = evaluate(best["model"], test_dl, loss_fn, device)
print(f"validation accuracy {best['val_acc']:.3f}   TEST accuracy {test_acc:.3f}")
assert test_acc > 0.75, f"Test accuracy {test_acc:.3f} is lower than expected. Check the earlier self-checks."
print("Test set used once. Do not tune on it anymore.")

## Stretch challenge (optional)

Add a **learning rate scheduler**, such as `torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)`, and pass it to `fit(..., scheduler=...)`. Plot the `lr` list from the history. Does it help? Write down what you saw:

*Your notes:*

## Reflection and submission

Write three lines in your portfolio repo's `module-03/README.md`:
1. What you built
2. What broke (and which self-check caught it)
3. What you would change next time

If you used wandb, add a link to your project page. Then download this notebook (*File > Download > .ipynb*), push it to `module-03` in your portfolio repo, and submit the link in Moodle.